In [58]:
####Simulation of the dataset####
import numpy as np
import pandas as pd

sample_size = 3000

df = pd.DataFrame({
    'roomMin': np.random.sample(sample_size),
    'roomMax': np.random.sample(sample_size),
    'priceMin': np.random.sample(sample_size),
    'priceMax': np.random.sample(sample_size)})


In [59]:
#####Modifying the complete simulated dataset#####

#percentage missing
#a for roomMin, b for roomMax, c for priceMin, d for priceMax
a = 0.23
b = 0.61
c = 0.69
d = 0.21

#create the 'missingness matrix'
df_NaN = pd.DataFrame({'NaN1' : np.random.sample(len(df)),
                       'NaN2' : np.random.sample(len(df)),
                       'NaN3' : np.random.sample(len(df)),
                       'NaN4' : np.random.sample(len(df))})

#transform the random values (uniform dist from 0 to 1) into integers referring to the missingness fraction
df_NaN.loc[(df_NaN.NaN1 <= a), 'NaN1'] = 0
df_NaN.loc[(df_NaN.NaN1 > a), 'NaN1'] = 1

df_NaN.loc[(df_NaN.NaN2 <= b), 'NaN2'] = 0
df_NaN.loc[(df_NaN.NaN2 > b), 'NaN2'] = 1

df_NaN.loc[(df_NaN.NaN3 <= c), 'NaN3'] = 0
df_NaN.loc[(df_NaN.NaN3 > c), 'NaN3'] = 1

df_NaN.loc[(df_NaN.NaN4 <= d), 'NaN4'] = 0
df_NaN.loc[(df_NaN.NaN4 > d), 'NaN4'] = 1

#multiply the missingness matrix with the dataset
df_with_missing = pd.DataFrame(df.values*df_NaN.values, columns=df.columns, index=df.index)

#remove rows where roomMin and roomMax or priceMin and priceMax are missing
Z = np.where((df_with_missing[df_with_missing.columns[0]] + df_with_missing[df_with_missing.columns[1]]==0) | (df_with_missing[df_with_missing.columns[2]] + df_with_missing[df_with_missing.columns[3]]==0))
Z1 = df_with_missing.drop(Z[0])
df_new = Z1.reset_index(drop = True)

#transform the 0 into NaN
df_new.loc[(df_new.roomMin == 0),'roomMin']= np.nan
df_new.loc[(df_new.roomMax == 0),'roomMax']= np.nan
df_new.loc[(df_new.priceMin == 0),'priceMin']= np.nan
df_new.loc[(df_new.priceMax == 0),'priceMax']= np.nan
df_new

,roomMin,roomMax,priceMin,priceMax
0,0.806130,NaN,NaN,0.139489
1,0.267024,0.946995,0.046715,0.575522
2,0.318501,NaN,0.624068,NaN
3,NaN,0.583810,NaN,0.533505
4,0.872774,0.350699,NaN,0.015707
...,...,...,...,...
2207,0.343758,NaN,NaN,0.905957
2208,0.041254,NaN,NaN,0.469073
2209,0.506194,0.147122,0.938868,0.381270
2210,NaN,0.471728,0.307257,0.156349


In [60]:
#####Preparing data for performance testing and iterative imputation#####
import random
seed = 30
np.random.seed(seed)
#splitting the dataset into two subsets
#df1 is for final performance testing (10% of the whole dataset); splitted into df1_store (5% of df1) for later comparison
#and df1_test (95% of df1) for actual actual performance testing
#df2 is for training and testing (90% of the whole dataset)
msk1 = np.random.rand(len(df_new)) < 0.1
df1 = df_new[msk1]
np.random.seed(seed + 1)
msk1_1 = np.random.rand(len(df1)) < 0.05
df1_store = df1[msk1_1]
df1_test = df1[~msk1_1]
df2 = df_new[~msk1]

#removing all rows with missing values in df2 (D0 in paper)
df2_full = df2.dropna()
np.random.seed(seed + 2)
#splitting into test (df2_test, D0,2 in paper) (20%) and training (df2_train, D0,1 in paper) (80%) set
msk2 = np.random.rand(len(df2_full)) < 0.8
df2_train = df2_full[msk2]
df2_test = df2_full[~msk2]
np.random.seed(seed + 3)
#second missingness matrix to determine which values from the test dataset (D0,2) to remove 
#to have the same missingness pattern
df_NaN2 = pd.DataFrame({'NaN1' : np.random.sample(len(df2_test)),
                        'NaN2' : np.random.sample(len(df2_test)),
                        'NaN3' : np.random.sample(len(df2_test)),
                        'NaN4' : np.random.sample(len(df2_test))})

df_NaN2.loc[(df_NaN2.NaN1 <= a), 'NaN1'] = 0
df_NaN2.loc[(df_NaN2.NaN1 > a), 'NaN1'] = 1

df_NaN2.loc[(df_NaN2.NaN2 <= b), 'NaN2'] = 0
df_NaN2.loc[(df_NaN2.NaN2 > b), 'NaN2'] = 1

df_NaN2.loc[(df_NaN2.NaN3 <= c), 'NaN3'] = 0
df_NaN2.loc[(df_NaN2.NaN3 > c), 'NaN3'] = 1

df_NaN2.loc[(df_NaN2.NaN4 <= d), 'NaN4'] = 0
df_NaN2.loc[(df_NaN2.NaN4 > d), 'NaN4'] = 1

#multiply the missingness matrix with the dataset
#df2_test_v1 (D0,2 mod in paper) is the test data set with the same missingness pattern as the original dataset (D in paper)
df2_test_v1 = pd.DataFrame(df2_test.values*df_NaN2.values, columns=df2_test.columns, index=df2_test.index)


#remove rows where roomMin and roomMax or priceMin and priceMax are missing and keeping the index
Y = df2_test_v1[(df2_test_v1[df2_test_v1.columns[0]] + df2_test_v1[df2_test_v1.columns[1]]==0) | (df2_test_v1[df2_test_v1.columns[2]] + df2_test_v1[df2_test_v1.columns[3]]==0)]
df2_test_new = df2_test_v1.drop(Y.index)

#transform the 0 into NaN
df2_test_new.loc[(df2_test_new.roomMin == 0),'roomMin']= np.nan
df2_test_new.loc[(df2_test_new.roomMax == 0),'roomMax']= np.nan
df2_test_new.loc[(df2_test_new.priceMin == 0),'priceMin']= np.nan
df2_test_new.loc[(df2_test_new.priceMax == 0),'priceMax']= np.nan

##at the beginning, there are already complete rows in the test set
##those rows have to added to the train set   
#dftest_complete contains the complete observations of the testset
indextoimpute_complete = df2_test_new[df2_test_new.isnull().sum(axis=1)!=0].index
index_incomplete = df2_test_new[df2_test_new.isnull().sum(axis=1)==0].index
dftest_complete = df2_test_new.drop(indextoimpute_complete)
#add it to the train set
df2_train = pd.concat([df2_full[msk2],dftest_complete])
#need another label for later iterations
df2_train_reset = pd.concat([df2_full[msk2],dftest_complete])
#update the test set
df2_test_new = df2_test_new.drop(index_incomplete)
#combination of df2_train and df2_test_new (for later performance testing)
df2_full1 = pd.concat([df2_train,df2_test_new])
df2_full_comp = df2_full.loc[df2_full1.index]

df2_test_new

,roomMin,roomMax,priceMin,priceMax
33,0.038320,NaN,NaN,0.172624
64,0.639170,0.293406,NaN,0.553020
76,0.939046,0.181441,NaN,0.719308
292,0.995801,0.840095,NaN,0.588574
367,0.237496,NaN,NaN,0.668920
543,0.392683,NaN,NaN,0.099204
590,0.795210,NaN,NaN,0.257124
1153,0.278960,NaN,NaN,0.821707
1188,NaN,0.536716,0.419711,0.995820
1200,0.523746,NaN,NaN,0.282085


In [61]:
#####K-Means Clustering#####

from sklearn.cluster import KMeans
min_clusters = 1
max_clusters = 5

#for the case we have division by zero
def division_0(n, d):
    return n / d if d else 0

#lists the NMSE (nomralized mean squared error) value for each run
NMSE_values_kmeans = []

i = 0
##imputing cluster centers into rows with i+1 missing values##
#shows which rows do not have i missing values (gives us an index)
indextoimpute = df2_test_new[df2_test_new.isnull().sum(axis=1)!=i+1].index
#we take the index from indextoimpute to get the dataframe with only i missing values (by dropping the rest)
dftest = df2_test_new.drop(indextoimpute)
dftest

,roomMin,roomMax,priceMin,priceMax
64,0.639170,0.293406,NaN,0.553020
76,0.939046,0.181441,NaN,0.719308
292,0.995801,0.840095,NaN,0.588574
1188,NaN,0.536716,0.419711,0.995820
1418,0.674559,0.081247,NaN,0.379269
1525,0.725638,NaN,0.765634,0.385289
1547,0.523593,NaN,0.647810,0.622112
2209,0.506194,NaN,0.938868,0.381270


In [62]:
##clustering##
#tells us which cells have missing values
missing = ~np.isfinite(dftest)
missing

,roomMin,roomMax,priceMin,priceMax
64,False,False,True,False
76,False,False,True,False
292,False,False,True,False
1188,True,False,False,False
1418,False,False,True,False
1525,False,True,False,False
1547,False,True,False,False
2209,False,True,False,False


In [63]:
#creating an array where each NaN=0 and each non-NaN keeps the value (transforms dataframe into an array)
imputation = np.where(missing, 0, dftest)
imputation

array([[0.63916955, 0.2934063 , 0.        , 0.55301999],
       [0.939046  , 0.18144071, 0.        , 0.71930809],
       [0.9958007 , 0.84009509, 0.        , 0.58857392],
       [0.        , 0.53671615, 0.4197113 , 0.9958196 ],
       [0.67455939, 0.08124653, 0.        , 0.37926947],
       [0.72563775, 0.        , 0.76563419, 0.3852891 ],
       [0.52359306, 0.        , 0.64780981, 0.62211174],
       [0.50619432, 0.        , 0.93886803, 0.38127029]])

In [64]:
#clusters the complete data
kmeans = KMeans(5, random_state=0).fit(df2_train)
result = kmeans.cluster_centers_
result

array([[0.20286604, 0.75595015, 0.63426287, 0.48532531],
       [0.37799468, 0.27855228, 0.46353712, 0.7818006 ],
       [0.77519404, 0.74513579, 0.33023797, 0.44566484],
       [0.74631206, 0.39170916, 0.72587825, 0.32523217],
       [0.19664318, 0.32685752, 0.31693845, 0.20930969]])

In [65]:
##finding the best cluster##

#array A contains all rows with i missing values
A = np.asarray(dftest)
#array B contains all clusters of one subset 
B = result
A

array([[0.63916955, 0.2934063 ,        nan, 0.55301999],
       [0.939046  , 0.18144071,        nan, 0.71930809],
       [0.9958007 , 0.84009509,        nan, 0.58857392],
       [       nan, 0.53671615, 0.4197113 , 0.9958196 ],
       [0.67455939, 0.08124653,        nan, 0.37926947],
       [0.72563775,        nan, 0.76563419, 0.3852891 ],
       [0.52359306,        nan, 0.64780981, 0.62211174],
       [0.50619432,        nan, 0.93886803, 0.38127029]])

In [91]:
A[7]

array([0.50619432,        nan, 0.93886803, 0.38127029])

In [66]:
#to store the arrays from the loop, will contain arrays with the manhattan distance of each row 
#to each cluster center
list = []
#loop: will go through all k rows with i+1 missing value
#calculating the distance: I will calculate the distance without prior imputation as it is easier and 
#the results are the same
#C calculates the distance, calculations with missing values result in 'NaN'
#D transforms 'NaN' in 0 and transforms the distance in absolute value
#finally, the manhattan distance is calculated by summarizing each array
for k in range(0,len(dftest)):
    C = A[k] - B
    D = np.absolute(np.nan_to_num(C))
    list.append(D.sum(axis=1, dtype='float'))

#this shows A[7]-result    
C

array([[ 0.30332828,         nan,  0.30460517, -0.10405502],
       [ 0.12819964,         nan,  0.47533092, -0.40053031],
       [-0.26899972,         nan,  0.60863006, -0.06439455],
       [-0.24011774,         nan,  0.21298978,  0.05603812],
       [ 0.30955114,         nan,  0.62192958,  0.1719606 ]])

In [67]:
D

array([[0.30332828, 0.        , 0.30460517, 0.10405502],
       [0.12819964, 0.        , 0.47533092, 0.40053031],
       [0.26899972, 0.        , 0.60863006, 0.06439455],
       [0.24011774, 0.        , 0.21298978, 0.05603812],
       [0.30955114, 0.        , 0.62192958, 0.1719606 ]])

In [92]:
#each value shows the distance to one cluster center
D.sum(axis=1, dtype='float')

array([0.71198847, 1.00406086, 0.94202433, 0.50914564, 1.10344133])

In [68]:
list

[array([0.96654205, 0.50480949, 0.69510913, 0.43323319, 0.8196879 ]),
 array([1.54467217, 0.72065539, 1.00119028, 0.79707831, 1.39781802]),
 array([0.9803282 , 1.3725755 , 0.45847505, 0.96121632, 1.69165931]),
 array([0.94427985, 0.51600868, 0.84804773, 1.12176137, 1.09914139]),
 array([1.25245282, 0.89640159, 0.83091927, 0.4362526 , 0.89348698]),
 array([0.75417925, 1.04625164, 0.54532824, 0.12048718, 1.15366973]),
 array([0.4710604 , 0.48955993, 0.74561972, 0.59766701, 1.07062329]),
 array([0.71198847, 1.00406086, 0.94202433, 0.50914564, 1.10344133])]

In [69]:
#list is containing n arrays with the manhattan distance to each cluster center
#the lowest value will be determined and 
#the number of the column (=label of the nearest cluster center) will be stored in an array (=E)
min_cols = lambda lst: [np.argmin(sublist) for sublist in lst]
E = np.asarray(min_cols(list))
E

array([3, 1, 2, 1, 3, 3, 0, 3], dtype=int64)

In [70]:
#result contains all clusters of one set
#result[E] contains an array which assigns each row one cluster 
#E assigns each missing value the nearest cluster
#result[[np.zeros(len(df2_test_new_iter),dtype=int)][missing] contains an array with all replaced values

#previously, imputation[missing] was the array with all missing values 
imputation[missing]

array([0., 0., 0., 0., 0., 0., 0., 0.])

In [71]:
#in this step, this array is replaced by an array with the values of the cluster center
imputation[missing] = result[E][missing]
imputation[missing]

array([0.72587825, 0.46353712, 0.33023797, 0.37799468, 0.72587825,
       0.39170916, 0.75595015, 0.39170916])

In [72]:
#now, imputation contains the rows with replaced values (as array)
#creating a new dataframe from the results
#from array to dataframe
df2_test_new_imputed = pd.DataFrame(data = imputation,
                                    index = dftest.index,
                                    columns = dftest.columns)
df2_test_new_imputed

,roomMin,roomMax,priceMin,priceMax
64,0.639170,0.293406,0.725878,0.553020
76,0.939046,0.181441,0.463537,0.719308
292,0.995801,0.840095,0.330238,0.588574
1188,0.377995,0.536716,0.419711,0.995820
1418,0.674559,0.081247,0.725878,0.379269
1525,0.725638,0.391709,0.765634,0.385289
1547,0.523593,0.755950,0.647810,0.622112
2209,0.506194,0.391709,0.938868,0.381270


In [73]:
#before:
dftest

,roomMin,roomMax,priceMin,priceMax
64,0.639170,0.293406,NaN,0.553020
76,0.939046,0.181441,NaN,0.719308
292,0.995801,0.840095,NaN,0.588574
1188,NaN,0.536716,0.419711,0.995820
1418,0.674559,0.081247,NaN,0.379269
1525,0.725638,NaN,0.765634,0.385289
1547,0.523593,NaN,0.647810,0.622112
2209,0.506194,NaN,0.938868,0.381270


In [74]:
#df2_train before:
df2_train

,roomMin,roomMax,priceMin,priceMax
14,0.701960,0.833319,0.237742,0.424611
32,0.053835,0.484478,0.091098,0.035869
54,0.427319,0.246841,0.554329,0.763532
66,0.204375,0.394368,0.759124,0.241123
80,0.167977,0.985637,0.483044,0.297261
...,...,...,...,...
2173,0.994226,0.094804,0.817797,0.675035
2184,0.386802,0.532818,0.893985,0.387127
2196,0.057598,0.543319,0.487863,0.422560
1649,0.259612,0.042535,0.641751,0.912493


In [75]:
##adding the imputed data to the train set (D(i+1)_imp in paper, i+1 stands for the number of imputations)   
frames = [df2_train, df2_test_new_imputed]
df2_train = pd.concat(frames)
df2_train

,roomMin,roomMax,priceMin,priceMax
14,0.701960,0.833319,0.237742,0.424611
32,0.053835,0.484478,0.091098,0.035869
54,0.427319,0.246841,0.554329,0.763532
66,0.204375,0.394368,0.759124,0.241123
80,0.167977,0.985637,0.483044,0.297261
...,...,...,...,...
1188,0.377995,0.536716,0.419711,0.995820
1418,0.674559,0.081247,0.725878,0.379269
1525,0.725638,0.391709,0.765634,0.385289
1547,0.523593,0.755950,0.647810,0.622112


In [76]:
##calculating NMSE
#storing the NMSE value of each column and number of imputed values per observation
sqrterror = []

v=0
#mean of the variable (for each column; mu in paper)
mu = np.mean(df2_train_reset[df2_train_reset.columns[v]])
mu

0.4806551970208997

In [83]:
#splitting the NMSE calculation
nominator = (df2_full_comp[df2_full_comp.columns[v]] - df2_train[df2_train.columns[v]])**2
denominator = df2_test_new.isna().sum()[v]*mu
nominator.fillna(0)

14      0.0
32      0.0
33      0.0
54      0.0
64      0.0
       ... 
2117    0.0
2173    0.0
2184    0.0
2196    0.0
2209    0.0
Name: roomMin, Length: 181, dtype: float64

In [85]:
#for the case that we have 0 replaced values in one column
F = division_0(nominator.fillna(0), denominator)
F

14      0.0
32      0.0
33      0.0
54      0.0
64      0.0
       ... 
2117    0.0
2173    0.0
2184    0.0
2196    0.0
2209    0.0
Name: roomMin, Length: 181, dtype: float64

In [86]:
sqrterror.append(np.sum(F))
np.sum(F)

0.011264835757603413

In [87]:
NMSE = np.sum(sqrterror)
NMSE

0.011264835757603413

In [ ]:
#####################

In [403]:
#####OPTICS Clustering#####
from sklearn.cluster import OPTICS

#for the case we have division by zero
def division_0(n, d):
    return n / d if d else 0

#lists the NMSE (nomralized mean squared error) value for each run
NMSE_values_optics = []

#OPTICS "finds core sample of high density and expands clusters from them"
#We iterate over the number of samples in a neighborhood for a point to be considered as a core point (fraction of sample size)
#example: 10%: a point will considered as high density point if at least 10% of the sample is in the neighborhood
#each sample gets a label for a cluster, except for noisy samples

#loop for finding the best fraction of the sample for the lowest NMSE 
for clusters in range(1, 20):
    #loop for the multiple imputation process
    for n in range(0,5):
        #I have to reset the dataframe df2_train, otherwise it would cumulate over all iterations
        df2_train = df2_train_reset
        #loop for the iterative imputation process
        for i in range(0,2):

            ##imputing cluster centers into rows with i missing values##
            #shows which rows do not have i missing values (gives us an index)
            indextoimpute = df2_test_new[df2_test_new.isnull().sum(axis=1)!=i+1].index
            #we take the index from indextoimpute to get the dataframe with only i missing values (by dropping the rest)
            dftest = df2_test_new.drop(indextoimpute)


            ##clustering##
            #tells us which cells have missing values
            missing = ~np.isfinite(dftest)
            #creating an array where each NaN=0 and each non-NaN keeps the value (transforms dataframe into an array)
            imputation = np.where(missing, 0, dftest)
            #clusters the complete data
            clustering = OPTICS(min_samples = clusters/100).fit(df2_train)
            optics = clustering.labels_
            optics_clusters = []
            #iterate for each label
            for labels in range(0,np.max(optics)+1):
                #look for the samples with the same label
                a = np.where(optics==labels)
                #take the mean of the samples with the same label to get the center of the cluster
                b = np.sum(core_array, axis=0)/np.max(np.shape(a))
                #store the cluster center
                optics_clusters.append(b)

            ##finding the best cluster##

            #array A contains all rows with i missing values
            A = np.asarray(dftest)
            #array B contains all clusters of one subset 
            B = np.asarray(optics_clusters)
            #to store the arrays from the loop, will contain arrays with the manhattan distance of each row 
            #to each cluster center
            list = []
            #loop: will go through all k rows with i+1 missing value
            #calculating the distance: I will calculate the distance without prior imputation as it is easier and 
            #the results are the same
            #C calculates the distance, calculations with missing values result in 'NaN'
            #D transforms 'NaN' in 0 and transforms the distance in absolute value
            #finally, the manhatten distance is calculated by summarizing each array
            for k in range(0,len(dftest)):
                C = A[k] - B
                D = np.absolute(np.nan_to_num(C))
                list.append(D.sum(axis=1, dtype='float'))

            #list is containing n arrays with the manhattan distance to each cluster center
            #the lowest value will be determined and 
            #the number of the column (=label of the nearest cluster center) will be stored in an array (=E)
            min_cols = lambda lst: [np.argmin(sublist) for sublist in lst]
            E = np.asarray(min_cols(list))

            #optics_clusters contains all clusters of one set
            #optics_clusters[E] contains an array which assigns each row one cluster 
            #E assigns each missing value the nearest cluster
            #optics_clusters[[np.zeros(len(df2_test_new_iter),dtype=int)][missing] contains an array with all replaced values

            #previously, imputation[missing] was the array with all missing values 
            #in this step, this array is replaced by an array with the values of the cluster center
            imputation[missing] = np.asarray(optics_clusters)[E][missing]

            #now, imputation contains the rows with replaced values (as array)
            #creating a new dataframe from the results
            #from array to dataframe
            df2_test_new_imputed = pd.DataFrame(data = imputation,
                                                index = dftest.index,
                                                columns = dftest.columns)

            ##adding the imputed data to the train set (D(i+1)_imp in paper, i+1 stands for the number of imputations)   
            frames = [df2_train, df2_test_new_imputed]
            df2_train = pd.concat(frames)
                
        #saving the dataset of the multiple imputation run n
        J = df2_train
        #after the 2nd run (n == 1): add the new dataset (=J) to the old one (=I, from the first run)
        #K is the cumulated dataset                         
        if n == 1:
            K = I + J
        #after the 3rd run +: add to the cumulated dataset of the previous run, the new dataset J     
        if n > 1:
            K = K + J
        #saving the dataset for the next run            
        I = df2_train        



    #the complete dataset is the result of 5 imputation runs, the datasets were added previously
    #now they are divided by 5
    estimated_df2 = K/5
    estimated_df2

    ##calculating NMSE
    #storing the NMSE value of each column and number of imputed values per observation
    sqrterror = []

    #v: number of column
    for v in range(0,len(estimated_df2.columns)):
        #mean of the variable before imputation (for each column; mu in paper)
        mu = np.mean(df2_train_reset[df2_train_reset.columns[v]])
        #splitting the NMSE calculation
        nominator = (df2_full_comp[df2_full_comp.columns[v]] - estimated_df2[estimated_df2.columns[v]])**2
        denominator = df2_test_new.isna().sum()[v]*mu
        #for the case that we have 0 replaced values in one column
        F = division_0(nominator, denominator)
        sqrterror.append(np.sum(F))
    NMSE = np.sum(sqrterror)
    NMSE_values_optics.append(NMSE)
NMSE_values_optics

[3.311250609156523,
 1.374598395104716,
 0.7661155353867387,
 0.6940352368820923,
 1.3228945644829695,
 1.9754709283521197,
 0.981299106392588,
 1.9754709283521197,
 1.9754709283521197,
 1.9754709283521197,
 1.9754709283521197,
 1.9754709283521197,
 1.9754709283521197,
 1.9754709283521197,
 1.9754709283521197,
 1.9754709283521197,
 1.9754709283521197,
 1.9754709283521197,
 1.9754709283521197]

In [404]:
#####matrix factorization#####
from sklearn.decomposition import NMF

for components in range(1,7):
    matrix = NMF(n_components = components, init="random", random_state = 0)
    W = matrix.fit_transform(np.nan_to_num(df2_full1))
    H = matrix.components_
W.dot(H)

array([[5.38244833e-02, 4.84510856e-01, 9.11030875e-02, 3.58659407e-02],
       [3.82786385e-02, 3.96202422e-01, 6.29478635e-01, 1.72624892e-01],
       [6.39221879e-01, 2.93323852e-01, 7.25383943e-01, 5.52997578e-01],
       [1.68004561e-01, 9.85626872e-01, 4.83053772e-01, 2.97242543e-01],
       [4.81818219e-01, 9.77447872e-01, 6.25659463e-01, 8.76701676e-01],
       [8.52444669e-01, 3.51702640e-01, 3.36092330e-03, 4.36478726e-01],
       [2.08845836e-01, 2.79439693e-01, 4.58417353e-01, 9.20510940e-01],
       [1.04482132e-02, 6.69067225e-01, 3.92912756e-01, 3.50761487e-01],
       [8.69076667e-02, 2.13440189e-01, 6.70462682e-01, 6.66259136e-01],
       [4.69046555e-01, 5.31682463e-01, 9.85614578e-01, 4.04357274e-01],
       [2.09273620e-01, 3.16144907e-01, 5.42769534e-01, 4.70762273e-01],
       [8.15109241e-01, 1.52642121e-01, 8.14307682e-01, 3.60235183e-01],
       [4.61655887e-01, 6.40061566e-01, 5.03567107e-01, 2.38785191e-01],
       [2.76581167e-02, 2.07944862e-02, 4.89104676e